# phase_2 detector on Kaggle — YOLOv8l, Drive-resumable

Trains the 29-region detector **exactly like the local run**, but checkpoints to
**Google Drive via rclone** so it survives Kaggle's ephemeral storage.

**Why Drive:** `/kaggle/working` is wiped between sessions unless you *Commit*, and
a session can die mid-run. We push the run dir to Drive **every `SYNC_EVERY`
optimizer steps + each epoch**; the next session pulls `last.pt` and `--resume`s.

**Each session:** Settings → Accelerator → **GPU**, then *Run All*. From the 2nd
session on, set `RESUME = 1` in CONFIG. That's the only change.

## CONFIG — edit these

In [ ]:
import os
# ===== Kaggle dataset slugs (edit) =====
PHASE2_SRC = "/kaggle/input/phase2-code/phase_2"        # uploaded phase_2 folder
IMAGES     = "/kaggle/input/mimic-cropped448"           # 448 cropped images (stem == image_id)
SCENE      = "/kaggle/input/mimic-scene-graph"          # *_SceneGraph.json (rescaled to 448)
META       = "/kaggle/input/mimic-metadata/mimic_metadata_final.jsonl"

# ===== training =====
MODEL    = "yolov8l.pt"   # same as local; auto-downloads (needs Internet ON)
IMGSZ    = 448            # images are 448
EPOCHS   = 50
RUN_NAME = "det29"
SUBSET   = 1000           # >0 = balanced N-image subset (like the local demo); 0 = FULL dataset
RESUME   = 0             # 0 = fresh; 1 = continue from the Drive checkpoint (2nd session on)

# ===== durable checkpoints (Google Drive via rclone) =====
RCLONE_REMOTE = "dhint:CHEX-DATA/phase2_runs"   # run dirs are pushed here
SYNC_EVERY    = 300                             # push every N optimizer steps
RCLONE_SECRET = "RCLONE_CONF"                   # Kaggle Secret holding your rclone.conf text

for k, v in dict(PHASE2_SRC=PHASE2_SRC, IMAGES=IMAGES, SCENE=SCENE, META=META, MODEL=MODEL,
                 IMGSZ=IMGSZ, EPOCHS=EPOCHS, RUN_NAME=RUN_NAME, SUBSET=SUBSET, RESUME=RESUME,
                 RCLONE_REMOTE=RCLONE_REMOTE, SYNC_EVERY=SYNC_EVERY, RCLONE_SECRET=RCLONE_SECRET).items():
    os.environ[k] = str(v)
print("config set | RESUME =", RESUME, "| SUBSET =", SUBSET)

## 1 — install rclone + write its config from a Kaggle Secret

**One-time setup:** locally run `rclone config file` to find your `rclone.conf`,
open it, copy the **entire text**. On Kaggle: *Add-ons → Secrets → Add a secret*,
label **`RCLONE_CONF`**, paste that text. (It contains your Drive token — keep the
notebook private.)

In [ ]:
%%bash
set -e
if ! command -v rclone >/dev/null 2>&1; then
  cd /kaggle/working
  curl -sLO https://downloads.rclone.org/rclone-current-linux-amd64.zip
  unzip -q -o rclone-current-linux-amd64.zip
  cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone
fi
rclone version | head -1

In [ ]:
import os, pathlib
# Graceful: if the RCLONE_CONF secret is missing or the remote is unreachable,
# training still runs — just WITHOUT Drive sync (checkpoints stay in /kaggle/working).
SYNC_OK = "0"
try:
    from kaggle_secrets import UserSecretsClient
    conf = UserSecretsClient().get_secret(os.environ["RCLONE_SECRET"])
    cfg = pathlib.Path.home() / ".config" / "rclone"; cfg.mkdir(parents=True, exist_ok=True)
    (cfg / "rclone.conf").write_text(conf)
    print("rclone.conf written")
    if os.system('rclone mkdir "%s"' % os.environ["RCLONE_REMOTE"]) == 0:
        SYNC_OK = "1"
        print("remote OK ->", os.environ["RCLONE_REMOTE"])
        os.system('rclone lsd "%s" 2>/dev/null | head' % os.environ["RCLONE_REMOTE"])
    else:
        print("[WARN] rclone could not reach the remote -> training WITHOUT Drive sync.")
except Exception as e:
    print("[WARN] RCLONE_CONF secret not set / rclone not ready:", e)
    print("       -> training will run WITHOUT Drive sync (checkpoints only in /kaggle/working).")
    print("       Enable later: Add-ons -> Secrets -> add RCLONE_CONF = your rclone.conf text.")
os.environ["SYNC_OK"] = SYNC_OK
print("SYNC_OK =", SYNC_OK)

## 2 — copy code into the writable dir + install ultralytics

In [ ]:
!rm -rf /kaggle/working/phase_2 && cp -r "$PHASE2_SRC" /kaggle/working/phase_2
%cd /kaggle/working/phase_2
!pip install -q ultralytics

## 3 — build the YOLO dataset

`SUBSET>0` picks a balanced N-image train/val set that has scene graphs (mirrors
the local demo). `SUBSET=0` uses the FULL metadata (all splits; gold→test).

In [ ]:
import os, json
S = "/kaggle/working"
META, SCENE = os.environ["META"], os.environ["SCENE"]
SUBSET = int(os.environ["SUBSET"])
META_USE = META

if SUBSET > 0:
    # list scene-graph dicoms (recursive), then pick N rows (85% train / 15% val) that have one
    import glob
    have = set()
    for fp in glob.iglob(os.path.join(SCENE, "**", "*_SceneGraph.json"), recursive=True):
        have.add(os.path.basename(fp)[:-len("_SceneGraph.json")])
    print("scene-graph dicoms:", len(have))
    def dicom(iid):
        p = iid.split("_", 3); return p[3] if len(p) == 4 else ""
    want = {"train": int(SUBSET * 0.85), "val": SUBSET - int(SUBSET * 0.85)}
    got = {"train": 0, "val": 0}; rows = []
    with open(META, encoding="utf-8-sig") as f:
        for line in f:
            r = json.loads(line)
            if str(r.get("dataset", "")).lower() != "mimic": continue
            sp = str(r.get("split", "")).lower()
            if sp not in want or got[sp] >= want[sp]: continue
            if dicom(r.get("image_id", "")) not in have: continue
            rows.append({"image_id": r["image_id"], "dataset": "mimic", "split": sp}); got[sp] += 1
            if all(got[k] >= want[k] for k in want): break
    META_USE = os.path.join(S, "meta_subset.jsonl")
    with open(META_USE, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r) + "\n")
    print("subset:", got)
os.environ["META_USE"] = META_USE

In [ ]:
!python build_yolo_dataset.py \
  --metadata "$META_USE" --images-root "$IMAGES" --scene-root "$SCENE" \
  --out /kaggle/working/yolo_ds --link-mode symlink

## 4 — train (Drive-synced)

Fresh run pushes checkpoints to Drive every `SYNC_EVERY` steps + each epoch.
If the session dies, set `RESUME=1` in CONFIG and *Run All* again — `train_yolo.py`
pulls `last.pt` from Drive and continues.

In [ ]:
import os
# add --sync-remote only if Drive is configured; --resume only if RESUME=1
os.environ["SYNC"] = ('--sync-remote "%s" --sync-every %s' %
                      (os.environ["RCLONE_REMOTE"], os.environ["SYNC_EVERY"])) \
                     if os.environ.get("SYNC_OK") == "1" else ""
os.environ["RESUME_FLAG"] = "--resume" if int(os.environ["RESUME"]) else ""
print("sync:", os.environ["SYNC"] or "(off)", "| resume:", os.environ["RESUME_FLAG"] or "(fresh)")
!python train_yolo.py \
  --data /kaggle/working/yolo_ds/dataset.yaml \
  --model "$MODEL" --device 0 --imgsz $IMGSZ --epochs $EPOCHS --batch -1 \
  --runs /kaggle/working/runs --name "$RUN_NAME" $SYNC $RESUME_FLAG

## 5 — eval mAP (val; use `--split test` for the gold set when SUBSET=0)

In [ ]:
!python eval_yolo.py \
  --weights /kaggle/working/runs/$RUN_NAME/weights/best.pt \
  --data /kaggle/working/yolo_ds/dataset.yaml --split val --device 0

## 6 — grab the trained model

`best.pt` is already on Drive (final push at train end). To pull it anywhere:

```bash
rclone copy dhint:CHEX-DATA/phase2_runs/det29/weights/best.pt ./   # local
```
Or download it from the Kaggle notebook's **Output** tab (`runs/det29/weights/best.pt`).

In [ ]:
import os
RUN = os.environ["RUN_NAME"]
if os.environ.get("SYNC_OK") == "1":
    os.system('rclone copy /kaggle/working/runs/%s "%s/%s" --quiet' % (RUN, os.environ["RCLONE_REMOTE"], RUN))
    print("pushed final run -> %s/%s" % (os.environ["RCLONE_REMOTE"], RUN))
else:
    print("Drive sync off -> download from the notebook 'Output' tab: runs/%s/weights/best.pt" % RUN)
!ls -lh /kaggle/working/runs/$RUN_NAME/weights/